# Protected-area Burmese text baseline and remediation audit

This notebook preserves the baseline review of `data/protected_areas.json`, explicitly excluding the already-known `တိရိစ္ဆာန်` issue. The 17 high-confidence records and 26 fields identified below were corrected on 2026-09-23. Non-standard `ဥ်/ဉ်` forms and review-only findings were intentionally left unchanged.

**Scope:** 39 records, 21 `_mm` fields, and 529 non-empty Burmese cells.


## Results

- Baseline: 17 protected areas contained high-confidence spelling, code-point, or repeated-text issues other than `တိရိစ္ဆာန်`.
- Remediation: all 26 identified record-field pairs were corrected in the JSON source and database.
- 23 protected areas separately contain specific words using non-standard `ဥ်` where standard spelling uses `ဉ်`.
- One additional duplicate list value (`ဆတ်`) and several proper-name/wording inconsistencies require editorial review rather than automatic classification.


In [ ]:
import json
from pathlib import Path

records = json.loads(Path('../data/protected_areas.json').read_text(encoding='utf-8-sig'))
fields = sorted({key for row in records for key in row if key.endswith('_mm')})
nonempty_cells = sum(bool(str(row.get(field) or '').strip()) for row in records for field in fields)
len(records), len(fields), nonempty_cells


## High-confidence patterns

The mapping below records the observed text and its standard or intended form. It deliberately excludes `တိရိစ္ဆာန်`.


In [ ]:
corrections = {
    'အာစီယံ': 'အာဆီယံ',
    'ဝါးမျိးစိတ်': 'ဝါးမျိုးစိတ်',
    'အခွင့်အလန်း': 'အခွင့်အလမ်း',
    'အမျိူးမျိုး': 'အမျိုးမျိုး',
    'ကုန်းနေရေနေနေ': 'ကုန်းနေရေနေ',
    'ဥယျာက်': 'ဥယျာဉ်',
    'တည်မှီး': 'တည်မှီ',
    'တောတိုနှင့်': 'တောတို့နှင့်',
    'တဆက်တည်း': 'တစ်ဆက်တည်း',
    'ထင်းရူးတော': 'ထင်းရှူးတော',
    'အပြင်မျိုးစိတ်': 'အပင်မျိုးစိတ်',
    'ဆေးဘက်ဝင်': 'ဆေးဖက်ဝင်',
    'အသင့်အတင်': 'အသင့်အတင့်',
    'ဖွံဖြိုးရေး': 'ဖွံ့ဖြိုးရေး',
    'ပါ၀င်': 'ပါဝင်',
    'ထိန်းသိမ်းရေးနယ်မြတစ်ခု': 'ထိန်းသိမ်းရေးနယ်မြေတစ်ခု',
    'အင်တိုင်းသစ်တော သစ်တောများ': 'အင်တိုင်းသစ်တောများ',
}

repeated_hlawga = (
    'လှော်ကားဥယျာဉ်သည် ရန်ကုန်မြို့နှင့် နီးကပ်သောကြောင့် '
    'ပြည်တွင်းခရီးသွားများနှင့် သဘာဝလေ့လာလိုသူများအတွက် '
    'လူကြိုက်များသော နေရာတစ်ခု ဖြစ်သည်။'
)

findings = []
for row in records:
    for field in fields:
        value = str(row.get(field) or '')
        for observed, expected in corrections.items():
            if observed in value:
                findings.append({
                    'slug': row['slug'], 'name_mm': row.get('name_mm'),
                    'field': field, 'observed': observed, 'expected': expected,
                    'count': value.count(observed),
                })
        if value.count(repeated_hlawga) == 2:
            findings.append({
                'slug': row['slug'], 'name_mm': row.get('name_mm'),
                'field': field, 'observed': 'full sentence repeated twice',
                'expected': 'keep one sentence', 'count': 1,
            })

len({item['slug'] for item in findings}), len({(item['slug'], item['field']) for item in findings}), sum(item['count'] for item in findings)


## Review-only findings

- The former duplicate `ဆတ်` finding in `hlawga_wildlife_park_protected.species_fauna_mm` was resolved on 2026-09-23 by retaining only the two animal records explicitly linked to Hlawga.
- `inlay_lake_wildlife_sanctuary_protected.tourism_mm` contains the semantically redundant phrase `နှစ်စဉ်နှစ်တိုင်း`.
- Proper-name differences in `sar_munm_national_park_protected.description_mm` (`စာမွန်း/စာမွှန်း`, `ကျီးရီးရမ်း/ကျီးရီးယန်း`) need an authoritative place-name source before correction.
- `inlay_lake_wildlife_sanctuary_protected.description_mm` contains the unclear wording `ရေစူးတိမ်`; this is a wording issue, not classified as a confirmed spelling error.


In [ ]:
u1_findings = [
    {'slug': row['slug'], 'name_mm': row.get('name_mm'),
     'fields': [field for field in fields if 'ဥ်' in str(row.get(field) or '')]}
    for row in records
    if any('ဥ်' in str(row.get(field) or '') for field in fields)
]
len(u1_findings)
